In [1]:
from huggingface_hub import HfFileSystem
from huggingface_hub import snapshot_download
from huggingface_hub import hf_hub_download
from multiprocess import Pool
import tarfile
import itertools
import json
from tqdm import tqdm
from glob import glob
from collections import defaultdict
import random
import os

fs = HfFileSystem()

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
files = fs.glob("datasets/amphion/Emilia-Dataset/Emilia-YODAS/*/*.tar")

In [3]:
splitted = [f.split('Emilia-Dataset/')[1] for f in files]
splitted[:2]

['Emilia-YODAS/DE/DE-B000000.tar', 'Emilia-YODAS/DE/DE-B000001.tar']

In [4]:
# !rm -rf done-download
# !mkdir done-download
# !mkdir Emilia-YODAS
!chmod -R 777 Emilia-YODAS

In [5]:
def loop(files):
    files, _ = files
    with open('Emilia-YODAS-Voice-Conversion-audio.json') as fopen:
        combined = set(json.load(fopen))
    for f in tqdm(files):
        done_filename = f"done-download/{f.replace('/', '_')}.json"
        try:
            with open(done_filename) as fopen:
                json.load(fopen)
                continue
        except:
            pass
            
        hf_hub_download(repo_id="amphion/Emilia-Dataset", filename=f, repo_type="dataset", local_dir="./")
        with tarfile.open(f, "r") as tar:
            tar.extractall(path=os.path.split(f)[0])
        os.remove(f)

        audio_files = glob(f'{os.path.split(f)[0]}/*.mp3')
        ins = [f_ for f_ in audio_files if f_ not in combined]
        # print(f, len(audio_files), len(ins))
        for f_ in ins:
            try:
                os.remove(f_)
                os.remove(f_.replace('.mp3', '.json'))
            except:
                pass
        
        with open(done_filename, 'w') as fopen:
            json.dump('done', fopen)

    del combined

In [6]:
# loop((splitted[:3], 0))

In [7]:
multiprocessing(splitted, loop, cores = 20, returned=False)

100%|██████████| 99/99 [2:29:44<00:00, 90.75s/it]  

100%|██████████| 99/99 [2:29:44<00:00, 90.75s/it]








100%|██████████| 99/99 [2:30:06<00:00, 90.97s/it]


In [11]:
json_files = glob('Emilia-YODAS/*/*.json')
len(json_files)

11365354